# A2 — Knowledge-Base Demo

Two authors, two sections, evidence for A2 form Section 5 ("Evidence It Works"):

- **Section 1 (below) — OCR quality.** Real Tesseract output scored against real gold labels
  in `grading_kit/labels.jsonl`, on the actual corpus. Not mocked, not a demo — this ran against
  real scanned pages.
- **Sections 2-6 — Stage 4 (chunk → embed → store — `index/chunk.py`, `index/embed.py`,
  `index/store.py`).** A real 3-page sample from the actual corpus, the real index pipeline,
  index statistics, one retrieval example, and one retrieval-level worst failure.
- **Section 7 — Person B placeholder** for the one remaining item: an OCR-level worst-failure
  example (OCR quality itself is now done in Section 1 below).

**Scope note for Sections 2-6:** they run against **3 real pages already OCR'd by Section 1**
(`chandalika_p0161`, `tin_sangi_p0218`, `arogya_p0053` — chosen for a reasonable line count and a
spread of OCR confidence/tier, out of the 10 gold-labelled pages Section 1 processes) — not the
full 437-page corpus, which is a separate, larger follow-up run (`scripts/get_data.sh` →
`scripts/build_index.sh`). The two former upstream blockers on the real
`pipeline.build_knowledge_base()` entry point are resolved: `ingest/enhance.py`'s VAE/diffusion
stage is a bonus feature not gated for this group's data speciality (E26 "dirty-ocr") and was
disabled via `configs/config.yaml`'s `enhance.enabled: false` rather than built; `governance/pii.py`
is now implemented for real. See `configs/design_choices.md`'s Stage 4 row and
`reports/pipeline_diagram.md` for the full picture. Sections 2-6 below are entirely real, unmocked
code and real, unmocked input — the actual `multilingual-e5-base` model, a real FAISS index, and
real OCR'd Bengali text with its actual confidence/evidence-tier values, not hand-written
placeholders.

## OCR Quality — CER / WER against the gold standard
Evaluates Stage 3's actual OCR accuracy against `grading_kit/labels.jsonl` (the independently
human-reviewed page sample) — a continuous accuracy measurement, distinct from the pass/fail
CER gate in `vision/ocr.py` (which only tags a line `"gold"` on an exact CER==0.0 match). A
line that fails that strict gate is still scored here, so this reports the real OCR error
rate, not just the gate's pass rate.

CER = character-level edit distance / reference length. WER is the same idea at word
granularity: WER = word-level edit distance / reference word count.

In [1]:
import sys
from pathlib import Path

# Find the project root and add the 'src' directory to sys.path
current = Path.cwd().resolve()
while current != current.parent:
    src_dir = current / "src"
    if src_dir.exists() and (src_dir / "doc_agent").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
    current = current.parent

import json
from doc_agent.ingest import loader, preprocess
from doc_agent.vision import layout, ocr
from doc_agent.vision.ocr import _normalize, _levenshtein, _align_lines

In [2]:
import json
import sys
from pathlib import Path

try:
    import yaml
except ImportError:
    !pip install pyyaml
    import yaml

# Find project root directory dynamically (searches upwards for configs or src)
current = Path.cwd().resolve()
project_root = current
while current != current.parent:
    if (current / "configs").exists() or (current / "src").exists():
        project_root = current
        break
    current = current.parent

# Load configuration relative to project root
config_path = project_root / "configs" / "config.yaml"

if config_path.exists():
    with open(config_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    print(f"Loaded configuration from: {config_path}")
else:
    cfg = {}
    print(f"Warning: Config file not found at {config_path}")

# Load gold standard labels relative to project root
labels_rel = cfg.get("grading_kit", {}).get("labels_path", "grading_kit/labels.jsonl")
labels_path = Path(labels_rel)
if not labels_path.is_absolute():
    labels_path = project_root / labels_path

gold_texts: dict[str, str] = {}
if labels_path.exists():
    with open(labels_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            row = json.loads(line)
            pid, text = row.get("page_id"), row.get("text", "")
            if pid and text and not text.startswith("REPLACE ME"):
                gold_texts[pid] = text

print(f"Loaded {len(gold_texts)} gold-labelled pages from {labels_path}")
if not gold_texts:
    print(
        "No gold labels yet -- fill grading_kit/labels.jsonl with reviewed page "
        "transcriptions before this section can report anything."
    )

Loaded configuration from: /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/configs/config.yaml
Loaded 10 gold-labelled pages from /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/grading_kit/labels.jsonl


In [3]:
# 1. Ensure configuration paths use resolved absolute paths
cfg.setdefault("paths", {})
cfg["paths"]["raw_dir"] = str((project_root / "data" / "raw").resolve())
cfg["paths"]["processed_dir"] = str(
    (project_root / "data" / "processed").resolve()
)

# 2. Guarantee the processed output directory exists
processed_dir = Path(cfg["paths"]["processed_dir"]).resolve()
processed_dir.mkdir(parents=True, exist_ok=True)

try:
    # 3. Run Stages 1-3 only for gold-labelled pages
    raw_pages = [p for p in loader.load_pages(cfg) if p.id in gold_texts]
    print(f"Raw pages: {len(raw_pages)}")
    pages = preprocess.run(raw_pages, cfg)
    regions = layout.detect(pages, cfg)
    ocr.transcribe(
        regions, cfg
    )  # writes data/processed/ocr_meta.jsonl + layout_meta.jsonl

    # 4. Parse OCR outputs safely using a context manager
    ocr_file = processed_dir / "ocr_meta.jsonl"
    ocr_rows = []
    if ocr_file.exists():
        with open(ocr_file, encoding="utf-8") as f:
            ocr_rows = [json.loads(line) for line in f if line.strip()]

    by_page: dict[str, list[str]] = {}
    for row in sorted(ocr_rows, key=lambda r: r["region_id"]):
        by_page.setdefault(row["page_id"], []).append(
            row["ocr_text_normalized"]
        )

    print(f"OCR'd {len(pages)} gold-labelled pages, {len(ocr_rows)} lines total")

except FileNotFoundError as e:
    by_page = {}
    print("Skipping -- corpus not available yet in this environment:")
    print(" ", e)

{"ts":"2026-08-13 18:30:54,737","lvl":"INFO","mod":"doc_agent.ingest.loader","msg":"loaded 833 pages across 6 documents from /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/raw"}
Raw pages: 10
{"ts":"2026-08-13 18:31:06,262","lvl":"INFO","mod":"doc_agent.ingest.preprocess","msg":"preprocessed 10 pages, dropped 0 blank/separator pages"}
{"ts":"2026-08-13 18:31:12,765","lvl":"INFO","mod":"doc_agent.vision.layout","msg":"detected 261 line regions across 10 pages -> /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/processed/layout_meta.jsonl"}
{"ts":"2026-08-13 18:31:28,061","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"OCR'd 261 regions across 10 pages: 0 accepted -> 0 chunks, 261 rejected -> /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/processed/ocr_meta.jsonl"}
OCR'd 10 gold-labelled pages, 261 lines total


In [4]:
# 3. Score CER/WER: line-level alignment reuses vision/ocr.py's own _align_lines -- the same
# content-aware DP alignment Stage 3's real accept/reject gate now uses (see vision/ocr.py), so this
# report's line-level scoring agrees with what actually got accepted/gold-tiered, instead of each
# maintaining its own separate notion of "aligned". An exact line-count match aligns positionally
# (unchanged); a mismatch of up to cfg['ocr']['max_line_diff'] (default 2) aligns by per-line
# similarity instead of skipping the page outright; a bigger mismatch is still skipped -- that
# page's line-level shape genuinely doesn't correspond 1:1 to the gold transcription (e.g. a prose
# paragraph transcribed as one gold "line" against a dozen wrapped visual lines), so line-level
# scoring for it stays honest rather than forcing a comparison. Also computes a page-level score so
# the notebook still reports meaningful OCR quality even for pages skipped at the line level.
import re

# Tesseract emits leading/trailing punctuation (danda ।, quotes, dashes, ...) as its own word-box,
# and vision/ocr.py::Reader._ocr_line() joins every detected word-box with a single space
# (" ".join(words)) regardless of whether a real visual gap existed -- so a correctly-read word
# immediately followed by correctly-read punctuation still comes out as "word ।" instead of "word।".
# CER already scores this fairly (it's a 1-character insertion, a small proportional penalty). WER
# does not: splitting purely on whitespace turns "word।" (one gold token) into two hyp tokens
# ["word", "।"], which registers as the word itself being WRONG *plus* an extra inserted token --
# one trivial spacing quirk can cost 2 full word-errors. _wer_tokenize() re-glues a punctuation-only
# suffix onto its preceding token before splitting, so WER measures reading accuracy instead of
# this spacing artifact. Verified against the real corpus (grading_kit/heldout_pages, 10 gold pages,
# 261 aligned lines): mean WER 0.3999 -> 0.3371 from this change alone, CER untouched (still exact,
# unnormalized-for-citation text -- this only affects word TOKENIZATION for the WER metric).
_WER_PUNCT_GLUE = re.compile(r"\s+(?=[।,;:.!?…—\-\"'“”‘’\)\]])")

def _wer_tokenize(text: str) -> list[str]:
    return _WER_PUNCT_GLUE.sub("", text).split()

def _wer(hyp_words: list[str], ref_words: list[str]) -> float:
    """Word Error Rate = word-level edit distance / reference word count."""
    if not ref_words:
        raise ValueError('_wer() requires a non-empty reference')
    return _levenshtein(hyp_words, ref_words) / len(ref_words)

line_results = []
page_results = []
skipped_pages = []
for page_id, gold_text in gold_texts.items():
    hyp_lines = by_page.get(page_id)
    if hyp_lines is None:
        continue
    ref_lines = [ln.strip() for ln in gold_text.splitlines() if ln.strip()]
    if not ref_lines:
        continue

    hyp_page_text = _normalize("\n".join(hyp_lines))
    ref_page_text = _normalize("\n".join(ref_lines))
    page_cer = _levenshtein(hyp_page_text, ref_page_text) / len(ref_page_text)
    ref_page_words = _wer_tokenize(ref_page_text)
    page_wer = _wer(_wer_tokenize(hyp_page_text), ref_page_words) if ref_page_words else None
    page_results.append({
        'page_id': page_id,
        'ocr_lines': len(hyp_lines),
        'gold_lines': len(ref_lines),
        'page_cer': page_cer,
        'page_wer': page_wer,
        'hyp_page_text': hyp_page_text,
        'ref_page_text': ref_page_text,
    })

    alignment = _align_lines(hyp_lines, ref_lines, page_id, "gold label")
    if not alignment:
        skipped_pages.append((page_id, len(hyp_lines), len(ref_lines)))
        continue

    for idx, ref in alignment.items():
        hyp = hyp_lines[idx]
        ref_norm = _normalize(ref)
        if not ref_norm:
            continue
        cer = _levenshtein(hyp, ref_norm) / len(ref_norm)
        hyp_words, ref_words = _wer_tokenize(hyp), _wer_tokenize(ref_norm)
        wer = _wer(hyp_words, ref_words) if ref_words else None
        line_results.append({
            'page_id': page_id,
            'cer': cer,
            'wer': wer,
            'hyp': hyp,
            'ref': ref_norm,
        })

print(f'scored {len(line_results)} lines across {len({r["page_id"] for r in line_results})} gold pages')
print(f'scored {len(page_results)} gold pages at page level')
if skipped_pages:
    print(f'skipped line-level scoring for {len(skipped_pages)} page(s) whose line-count gap is too '
          f'large to align (ocr lines vs. gold lines): {skipped_pages}')

{"ts":"2026-08-13 18:31:29,852","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"page chandalika_p0161: gold label has 27 lines vs. 26 detected regions — fuzzy-aligned 26/26 lines (difference <= 2)"}
{"ts":"2026-08-13 18:31:31,677","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"page tin_sangi_p0219: gold label has 33 lines vs. 31 detected regions — fuzzy-aligned 31/31 lines (difference <= 2)"}
{"ts":"2026-08-13 18:31:32,908","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"page tin_sangi_p0211: gold label has 25 lines vs. 23 detected regions — fuzzy-aligned 23/23 lines (difference <= 2)"}
{"ts":"2026-08-13 18:31:33,036","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"page chitrangada_p0152: gold label has 18 lines vs. 16 detected regions — fuzzy-aligned 16/16 lines (difference <= 2)"}
{"ts":"2026-08-13 18:31:34,897","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"page tin_sangi_p0216: gold label has 31 lines vs. 29 detected regions — fuzzy-aligned 29/29 lines (difference <= 2)"}
{"

In [5]:
# 4. Display aggregate CER / WER
import pandas as pd

if line_results:
    mean_cer = sum(r['cer'] for r in line_results) / len(line_results)
    wer_values = [r['wer'] for r in line_results if r['wer'] is not None]
    mean_wer = sum(wer_values) / len(wer_values) if wer_values else float('nan')
    print(f'Line-level mean CER: {mean_cer:.4f}')
    print(f'Line-level mean WER: {mean_wer:.4f}')
    df = pd.DataFrame(line_results)[['page_id', 'cer', 'wer', 'hyp', 'ref']]
    df
else:
    print('No pages had exact line-count matches, so line-level scoring was skipped.')

if page_results:
    mean_page_cer = sum(r['page_cer'] for r in page_results) / len(page_results)
    page_wer_values = [r['page_wer'] for r in page_results if r['page_wer'] is not None]
    mean_page_wer = sum(page_wer_values) / len(page_wer_values) if page_wer_values else float('nan')
    print(f'Page-level mean CER: {mean_page_cer:.4f}')
    print(f'Page-level mean WER: {mean_page_wer:.4f}')
    page_df = pd.DataFrame(page_results)[['page_id', 'ocr_lines', 'gold_lines', 'page_cer', 'page_wer']]
    page_df
else:
    print('Nothing scored -- check that grading_kit/labels.jsonl has real entries and that '
          'the corresponding pages exist under data/raw/.')

Line-level mean CER: 0.0967
Line-level mean WER: 0.3371
Page-level mean CER: 0.0953
Page-level mean WER: 0.3072


In [6]:
import sys
import tempfile
from pathlib import Path

import yaml

# Make src/ importable -- Section 1 above already does this, but Section 2 is written to also work
# standalone (e.g. if a reader re-runs just from here), so it repeats the same defensive check
# rather than assuming Section 1 already ran in this kernel.
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from doc_agent.contracts import Chunk  # noqa: E402
from doc_agent.index import chunk, embed, store  # noqa: E402

# Load the REAL project config: the MODEL choices below (multilingual-e5-base, FAISS inner-product
# search) are exactly what scripts/build_index.sh would use for the real corpus. Two SCALE-
# dependent settings are overridden just below for this small real sample, each with its own
# reason -- everything else (embed model/dim, base index metric) is untouched. Named demo_cfg (not
# cfg) so it never collides with Section 1's own `cfg` variable in this shared notebook kernel.
with open(ROOT / "configs" / "config.yaml", encoding="utf-8") as f:
    demo_cfg = yaml.safe_load(f)

demo_dir = Path(tempfile.mkdtemp(prefix="kb_demo_"))
demo_cfg["paths"] = {
    "raw_dir": str(demo_dir / "raw"),
    "processed_dir": str(demo_dir / "processed"),
    "index_dir": str(demo_dir / "processed" / "index"),
}
Path(demo_cfg["paths"]["processed_dir"]).mkdir(parents=True, exist_ok=True)

# config.yaml's production index type is "faiss:hnsw" (M=32 graph links per node) -- correct for
# the real ~1000s-of-chunks corpus, but HNSW needs a graph with enough nodes to be meaningful, and
# this small sample produces only a couple dozen chunks. Confirmed directly (not assumed):
# faiss-cpu's IndexHNSWFlat SEGFAULTS in this environment when .add() is called with fewer vectors
# than M -- not a "bug in this notebook," a real rough edge in a graph index built for far more
# data than a small sample has. store.py's own test suite (tests/test_retrieval.py) already avoids
# this the same way, using "faiss:flat" for every test. Same fix here, same reason.
demo_cfg["index"]["type"] = "faiss:flat"

# config.yaml's production chunk_tokens/overlap (256/32) are sized for the real 437-page corpus --
# at that size, this sample's ~550 tokens would merge into 1-2 chunks, which can't demonstrate
# index statistics across chunks or a retrieval choice among candidates (Sections 4-5 below need
# more than a couple chunks to be meaningful). Scaled down for THIS sample's size only; the real
# corpus run still uses config.yaml's 256/32 unmodified.
demo_cfg["index"]["chunk_tokens"] = 40
demo_cfg["index"]["overlap"] = 8

# 3 real pages out of Section 1's 10 gold-labelled pages -- picked after inspecting Section 1's
# real by_page/ocr_rows output for a reasonable line count and a spread of OCR confidence/tier
# (not the first N pages found; see the per-page breakdown Section 1 produces above). Each is a
# different literary work, so this sample also exercises cross-document chunk isolation.
SAMPLE_PAGE_IDS = ["chandalika_p0161", "tin_sangi_p0218", "arogya_p0053"]

print(f"sample pages: {SAMPLE_PAGE_IDS}")
print(f"embed model: {demo_cfg['embed']['model']} (dim={demo_cfg['embed']['dim']})")
print(
    f"index type: {demo_cfg['index']['type']} (sample-scale override, see comment above; "
    f"config.yaml's real value is 'faiss:hnsw')"
)
print(
    f"chunk_tokens={demo_cfg['index']['chunk_tokens']}, overlap={demo_cfg['index']['overlap']} "
    f"(sample-scale override; config.yaml's real values are 256/32)"
)

sample pages: ['chandalika_p0161', 'tin_sangi_p0218', 'arogya_p0053']
embed model: intfloat/multilingual-e5-base (dim=768)
index type: faiss:flat (sample-scale override, see comment above; config.yaml's real value is 'faiss:hnsw')
chunk_tokens=40, overlap=8 (sample-scale override; config.yaml's real values are 256/32)


## 2. The real 3-page sample

The 88 real OCR'd lines Section 1 already produced for `chandalika_p0161`, `tin_sangi_p0218`, and
`arogya_p0053` — pulled directly from Section 1's own `ocr_rows` (the actual rows
`vision/ocr.py::transcribe()` wrote to `ocr_meta.jsonl`: `chunk_id`, `ocr_confidence`,
`evidence_tier`, `ocr_text_normalized`). No text is written by hand here — every line's text,
confidence, and tier below is exactly what Stage 3's real Tesseract pass produced, re-saved into a
scratch `ocr_meta.jsonl` sidecar in the *same row shape* `chunk.split()` expects, so the unmodified
function below can't tell the difference from a real corpus run.

In [7]:
import json

# Pull the real rows for the 3 sample pages straight out of Section 1's own `ocr_rows` -- the
# exact rows vision/ocr.py::transcribe() wrote to data/processed/ocr_meta.jsonl for the real
# corpus. No hand-written text, no invented confidence/tier values.
sample_rows = sorted(
    (r for r in ocr_rows if r["page_id"] in SAMPLE_PAGE_IDS),
    key=lambda r: (r["page_id"], r["region_id"]),
)
if not sample_rows:
    raise RuntimeError(
        "No OCR rows found for SAMPLE_PAGE_IDS -- Section 1 above must run successfully first "
        "(same notebook kernel) before this cell can pull real OCR output from it."
    )

line_chunks = []
ocr_meta_rows = []
for r in sample_rows:
    doc_id = r["page_id"].rsplit("_p", 1)[0]  # e.g. "chandalika_p0161" -> "chandalika"
    line_chunks.append(
        Chunk(id=r["chunk_id"], doc_id=doc_id, text=r["ocr_text_normalized"], page_ids=[r["page_id"]])
    )
    ocr_meta_rows.append(
        {
            "chunk_id": r["chunk_id"],
            "ocr_confidence": r["ocr_confidence"],
            "evidence_tier": r["evidence_tier"],
        }
    )

# Written under demo_cfg["paths"]["processed_dir"] -- exactly where the real ocr.transcribe() would
# have written it, so chunk.split() (called unmodified below) can't tell the difference.
ocr_meta_path = Path(demo_cfg["paths"]["processed_dir"]) / "ocr_meta.jsonl"
with open(ocr_meta_path, "w", encoding="utf-8") as f:
    for row in ocr_meta_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

n_pages = len({c.page_ids[0] for c in line_chunks})
n_docs = len({c.doc_id for c in line_chunks})
print(f"{len(line_chunks)} real OCR'd line-chunks across {n_pages} pages, {n_docs} documents: {SAMPLE_PAGE_IDS}")

87 real OCR'd line-chunks across 3 pages, 3 documents: ['chandalika_p0161', 'tin_sangi_p0218', 'arogya_p0053']


## 3. The real Stage 4 chain: chunk → embed → store

From here on, nothing is mocked — this calls the actual `index/chunk.py`, `index/embed.py`, and
`index/store.py` functions, the same ones `scripts/build_index.sh` calls for the real corpus. The
first call to `embed.encode()` downloads `intfloat/multilingual-e5-base` (~1.1GB) from the
Hugging Face Hub if it isn't already cached locally — one-time cost, cached for every future run.

In [8]:
index_chunks = chunk.split(line_chunks, demo_cfg)
print(f"chunk.split(): {len(line_chunks)} lines -> {len(index_chunks)} merged chunk(s)")
for c in index_chunks:
    print(f"  {c.id}  pages={c.page_ids}  text={c.text[:60]!r}...")

vectors = embed.encode(index_chunks, demo_cfg)
print(f"\nembed.encode(): shape={vectors.shape}, dtype={vectors.dtype}")

store.build(index_chunks, vectors, demo_cfg)
faiss_index, chunk_rows = store.load(demo_cfg)
print(
    f"\nstore.build()/load(): {faiss_index.ntotal} vectors indexed, "
    f"{len(chunk_rows)} chunk records loaded back"
)

{"ts":"2026-08-13 18:31:37,287","lvl":"INFO","mod":"doc_agent.index.chunk","msg":"re-chunked 87 lines into 19 index chunks (chunk_tokens=40, overlap=8)"}
chunk.split(): 87 lines -> 19 merged chunk(s)
  arogya_c00000  pages=['arogya_p0053']  text='আরোগা ৫৫ তাহাঁর। দিয়েছে মৌরে সৌভাগ্যের শেষ পরিচয়, ভুলায়ে '...
  arogya_c00001  pages=['arogya_p0053']  text='আছে জীবনের শ্রেষ্ঠ যেই দান । সমস্ত জ।বন ধরে খ্যাতির খাঁজন। দ'...
  arogya_c00002  pages=['arogya_p0053']  text='জানুয়ারি, ১৯৪১ । সকাল ১৬ দিন পরে যায় দিন, স্তব্ধ বসে থাকি;'...
  arogya_c00003  pages=['arogya_p0053']  text='যাহা, কী ধিয়েছি যাহ! ছিল দেয়, কী রয়েছে শেষের পাথেয় । যাঁ'...
  arogya_c00004  pages=['arogya_p0053']  text='আজ বাজিছে বৃথাই, হয়তো হয় নি জান। ক্ষমা করে কে গিয়েছে চলে '...
  arogya_c00005  pages=['arogya_p0053']  text='ছিন্ন হল জীবনের আঁন্তরণময়, জোড়া লাগাঁবারে আর রবে ন। সময় ।'...
  chandalika_c00000  pages=['chandalika_p0161']  text='১৬৮ রবীক্দ্র-রচনাবলী আধার অঙ্গনে প্রদীপ জালি নি, দগ্ধ কাঁননে'...
  chanda

/home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{"ts":"2026-08-13 18:31:40,769","lvl":"INFO","mod":"doc_agent.index.embed","msg":"loading embedding model intfloat/multilingual-e5-base on cpu"}
{"ts":"2026-08-13 18:31:50,140","lvl":"INFO","mod":"doc_agent.index.embed","msg":"embedded 19 chunks with intfloat/multilingual-e5-base (dim=768)"}

embed.encode(): shape=(19, 768), dtype=float32
{"ts":"2026-08-13 18:31:50,167","lvl":"INFO","mod":"doc_agent.index.store","msg":"built faiss:flat index with 19 vectors (dim=768) -> /tmp/kb_demo_o3fmkxke/processed/index"}
{"ts":"2026-08-13 18:31:50,168","lvl":"INFO","mod":"doc_agent.index.store","msg":"loaded index with 19 vectors and 19 chunk records from /tmp/kb_demo_o3fmkxke/processed/index"}

store.build()/load(): 19 vectors indexed, 19 chunk records loaded back


## 4. Index statistics (real numbers, real 3-page sample)

Pulled live from the `faiss_index`/`chunk_rows` built in Section 2 — nothing hardcoded. Real counts
for what was actually indexed here (3 real pages, not the full 437-page corpus — see the scope note
in the title cell for why the full run is a separate follow-up).

In [9]:
from collections import Counter

n_chunks = faiss_index.ntotal
dim = vectors.shape[1]
index_type = demo_cfg["index"]["type"]
distinct_docs = {row["doc_id"] for row in chunk_rows}
tier_counts = Counter(row["tier"] for row in chunk_rows)
total_pages_covered = {pid for row in chunk_rows for pid in row["page_ids"]}

print(f"chunks indexed:        {n_chunks}")
print(f"embedding dimension:   {dim}")
print(f"index type (sample):   {index_type}  (real corpus run uses config.yaml's 'faiss:hnsw')")
print(f"distinct doc_ids:      {len(distinct_docs)}  ({sorted(distinct_docs)})")
print(f"tier breakdown:        {dict(tier_counts)}")
print(
    f"corpus coverage:       {len(total_pages_covered)} real pages "
    f"({sorted(total_pages_covered)}) -- a sample, not the full 437-page corpus"
)

# Worth noting honestly: chunk.split()'s tier aggregation is "gold" only if EVERY constituent line
# is gold, else "silver" -- there is no chunk-level "raw". So a chunk-level "silver" here doesn't
# necessarily mean any of its lines were individually silver-tiered -- printed below, not assumed,
# since ocr.py's own gold/raw split for a given line can vary run to run (its gate isn't this
# stage's file to explain) and this cell should stay accurate regardless of which lines land where.
line_tiers = {r["evidence_tier"] for r in sample_rows}
print(
    f"\n(for context: the underlying LINE-level tiers actually observed in this run's sample "
    f"were {line_tiers} -- chunk-level tiers above are the conservative per-chunk aggregate of "
    f"whatever those turned out to be, not an independent re-measurement)"
)

chunks indexed:        19
embedding dimension:   768
index type (sample):   faiss:flat  (real corpus run uses config.yaml's 'faiss:hnsw')
distinct doc_ids:      3  (['arogya', 'chandalika', 'tin_sangi'])
tier breakdown:        {'silver': 19}
corpus coverage:       3 real pages (['arogya_p0053', 'chandalika_p0161', 'tin_sangi_p0218']) -- a sample, not the full 437-page corpus

(for context: the underlying LINE-level tiers actually observed in this run's sample were {'raw'} -- chunk-level tiers above are the conservative per-chunk aggregate of whatever those turned out to be, not an independent re-measurement)


## 5. One real retrieval example (direct FAISS, not `retrieval/retriever.py`)

`retrieval/retriever.py` is A3 scope and still `raise NotImplementedError` by design (see
`reports/pipeline_diagram.md` — its `top_score()`/`is_weak()`/`next_k()` helpers are filled in as
A3 plumbing, but `Retriever.retrieve()` itself is not built yet). This cell demonstrates retrieval
directly against the index built above instead: `store.load()` + a query embedding +
`faiss_index.search()`.

The query below is a real phrase copied verbatim out of one real chunk's own OCR'd text (not
paraphrased, not hand-written) — chosen because it's from the *interior* of a chunk, not its
opening (Section 6 below covers what happens at a chunk's opening, where the sliding-window
overlap changes the picture).

One deliberate detail: `embed.encode()` always applies e5's `"passage: "` prefix (it's written for
indexing text, not querying it) — e5's asymmetric scheme expects `"query: "` on the query side
instead. `retriever.py` will own that logic for real in A3; here, for this one illustrative query,
the prefix is applied manually via the same cached model `embed._load_model()` returns, rather than
duplicating retriever.py's future responsibility.

In [10]:
# A real phrase, copied verbatim from the interior of chunk chandalika_c00002's actual OCR'd text
# (part of the "flower" personification passage in Chandalika) -- not paraphrased, not hand-written.
query_text = "দেবতা ওগো তোমার সেবা"

model = embed._load_model(demo_cfg["embed"]["model"], embed._resolve_device(demo_cfg))
query_vector = model.encode([f"query: {query_text}"], normalize_embeddings=True).astype("float32")

k = min(3, faiss_index.ntotal)
scores, ids = faiss_index.search(query_vector, k)

print(f"query: {query_text!r}\n")
for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), start=1):
    row = chunk_rows[idx]
    print(f"#{rank}  score={score:.4f}  id={row['id']}  pages={row['page_ids']}")
    print(f"      text: {row['text']}\n")

top = chunk_rows[ids[0][0]]
right_page = "chandalika_p0161" in top["page_ids"]
print(f"Top hit on the right page (chandalika_p0161, where this phrase actually appears)? {right_page}")
print(
    "Note this also beats chunks from the other two, unrelated pages/documents in this sample -- "
    "a real cross-document semantic-discrimination check, not just a same-page ranking."
)

{"ts":"2026-08-13 18:31:50,187","lvl":"WARNING","mod":"doc_agent.index.embed","msg":"cfg['device']='cuda' but no CUDA device is available -- falling back to cpu"}
query: 'দেবতা ওগো তোমার সেবা'

#1  score=0.8206  id=chandalika_c00002  pages=['chandalika_p0161']
      text: প্রস্থান ফুল বলে, ধন্য আমি ধন্য অবমি মাটির 'পরে। দেবতা ওগো, তোমার সেবা আমার খবে। জন্ম নিয়েছি ধুলিতে, দয়া করে দাঁও ভুলিতে, নাই ধূলি মোর অস্তরে ! নয়ন তোমার নত করো, দূলগুলি কাঁপে থরোথরো ।

#2  score=0.7951  id=arogya_c00003  pages=['arogya_p0053']
      text: যাহা, কী ধিয়েছি যাহ! ছিল দেয়, কী রয়েছে শেষের পাথেয় । যাঁর। কাছে এসেছিল, যাঁর! চলে গিয়েছিল দূরে, তাঁদের পরশখানি রয়ে গেছে মোর কোন্‌ স্থবে। অন্যমনে কারে চিনি নাই, বিদাঁয়ের পদধ্বনি প্রাণে আজ বাজিছে বৃথাই, হয়তো হয় নি জান। ক্ষমা

#3  score=0.7900  id=tin_sangi_c00000  pages=['tin_sangi_p0218']
      text: কিছুতে ঘে লজ্জার কারণ আছে তা যেন ও জানেই না। এই ওর অকৃত্রিম অবিবেক,. এই ঘে উচিত-অহ্থচিতের বেড়া অনায়াসে লাঁফ দিয়ে ভিডিয়ে চলা, এতেই, মেয়েদের স্নেহ ওকে এত 

## 6. Worst failure — retrieval-level (index/chunk/embed/store's own evidence)

A concrete, real retrieval failure demonstrated on the real 3-page index built above — the
OCR-level worst failure is a separate placeholder for Person B, see the final section below.

In [11]:
# The query is that chunk's OWN opening words, taken verbatim -- not hand-written, not a
# paraphrase, and not cherry-picked for effect: this was found by testing every one of the 18 real
# chunks' own opening 5 words as a query against this same index and checking which ones failed to
# retrieve themselves as top-1. tin_sangi_c00002 was the one real failure found this way.
target_id = "tin_sangi_c00002"
target_row = next(row for row in chunk_rows if row["id"] == target_id)
query_text = " ".join(target_row["text"].split()[:5])

query_vector = model.encode([f"query: {query_text}"], normalize_embeddings=True).astype("float32")
scores, ids = faiss_index.search(query_vector, faiss_index.ntotal)

print(f"query: {query_text!r}  (verbatim opening of chunk {target_id})\n")
for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), start=1):
    row = chunk_rows[idx]
    flag = "  <-- expected top-1, but isn't" if row["id"] == target_id else ""
    print(f"#{rank}  score={score:.4f}  id={row['id']}{flag}")
    print(f"      text: {row['text']}\n")

winner_id = chunk_rows[ids[0][0]]["id"]
print(f"Top hit is {winner_id!r}, not {target_id!r}.")

# Why, verified directly rather than guessed: chunk_tokens=40/overlap=8 means the LAST 8 tokens of
# one window are shared verbatim with the FIRST 8 tokens of the next. target_id's own opening 5
# words are entirely inside that shared 8-token overlap -- so the exact queried phrase is not
# unique to target_id, it's also verbatim present at the END of its immediate predecessor chunk.
winner_row = next(row for row in chunk_rows if row["id"] == winner_id)
shared = query_text in winner_row["text"]
print(f"\nIs the query phrase also present verbatim inside {winner_id!r}'s own text? {shared}")
if shared:
    tail = winner_row["text"][winner_row["text"].find(query_text):]
    print(f"  {winner_id} ends with: ...{tail!r}")

query: 'নীল পেনসিলের দাঁগ-কাটাঁকাঁটি করে শেষকালে'  (verbatim opening of chunk tin_sangi_c00002)

#1  score=0.8159  id=tin_sangi_c00001
      text: করবাঁর জোর পায় না। কর্তব্যবোঁধকে যারা অত্যন্ত সামলে চলে মেয়ের! তাদের পায়ের ধুলে! নেয় । আর যে-সব দুর্দাম দুরস্তের কোনে। বালাই নেই ন্যায়-অন্যায়ের, মেয়ের! তাঁদের বাহুবন্ধনে বাঁধে । ভেস্কের ব্লটিউকাগজটার উপর খানিকক্ষণ নীল পেনসিলের দাঁগ-কাটাঁকাঁটি করে শেষকালে বিভ। বললে, “আচ্ছা,

#2  score=0.8100  id=tin_sangi_c00002  <-- expected top-1, but isn't
      text: নীল পেনসিলের দাঁগ-কাটাঁকাঁটি করে শেষকালে বিভ। বললে, “আচ্ছা, যদি আমার হাতে টাকা থাকে তবে অমনি তোমাকে দেব। কিন্তু তোমার ওই ঘড়ি আমি কিছুতেই কিনব না ।” উত্তেজিত কঠে অভীক বললে, “ভিক্ষা? তোমার সমান ধনী যদি হতুম, তা হলে তোমার দান

#3  score=0.7839  id=arogya_c00003
      text: যাহা, কী ধিয়েছি যাহ! ছিল দেয়, কী রয়েছে শেষের পাথেয় । যাঁর। কাছে এসেছিল, যাঁর! চলে গিয়েছিল দূরে, তাঁদের পরশখানি রয়ে গেছে মোর কোন্‌ স্থবে। অন্যমনে কারে চিনি নাই, বিদাঁয়ের পদধ্বনি প্রাণে আজ বাজিছে বৃথাই, হয়তো হয় 

**Read on why:** this is not the "short, generic query" failure mode one might expect — the
queried phrase is a specific, non-generic clause, not a pronoun. The real cause, confirmed directly
above by checking for the substring rather than assumed: `chunk.split()`'s sliding window
(`chunk_tokens=40`, `overlap=8`) makes the last 8 tokens of one chunk identical to the first 8
tokens of the next, so a query built from a chunk's own opening words is, by construction, *also*
verbatim present inside its immediate predecessor. `multilingual-e5-base` has no way to prefer the
chunk where the phrase happens to open over the one where it happens to close — both genuinely
contain the exact queried text, and the embedding similarity difference between them comes down to
whatever surrounding context happens to shift the vector slightly, not to which chunk "really"
contains the phrase.

This is a structural consequence of chunking with overlap, not a one-off fluke, an artifact of this
small sample's scale, or specific to Bengali — the same ambiguity exists at the real corpus's
production `chunk_tokens=256`/`overlap=32`, just diluted across more distinguishing content per
chunk. It's exactly the kind of case `configs/design_choices.md`'s Stage 5 note and the A2 form's
Section 6 flag as an open risk for A3: whether reranking (`cfg.retrieve.rerank`) or evidence-gated
re-search (widen `k` on weak top-score, per `retriever.py`'s already-stubbed `is_weak()`/`next_k()`)
resolves boundary-straddling ambiguity like this, or whether citing the *source line IDs*
(`chunk_meta.jsonl`'s `source_line_ids`, which are NOT shared between overlapping chunks even when
their merged text is) is the more reliable disambiguator at answer time.

## 7. OCR-level worst failure (real, from Section 1's own scored lines)

Sorts Section 1's real `line_results` by `cer` descending and reports what's actually there,
rather than assuming it must be a conjunct-consonant (যুক্তাক্ষর) / matra segmentation issue just
because that's the likeliest risk category `configs/design_choices.md`'s Stage 3 note flags —
checked below, not assumed.

In [12]:
worst_lines = sorted(line_results, key=lambda r: -r["cer"])

print("Top 5 lines by CER (worst first):\n")
for r in worst_lines[:5]:
    print(f"page={r['page_id']}  cer={r['cer']:.4f}")
    print(f"  hyp: {r['hyp']}")
    print(f"  ref: {r['ref']}\n")

# The literal single worst line by CER, whatever it turns out to be this run -- reported honestly
# rather than skipped in favour of a more "interesting" one further down the list.
literal_worst = worst_lines[0]
print(f"Literal worst line: page={literal_worst['page_id']}  cer={literal_worst['cer']:.4f}")
print(f"  hyp: {literal_worst['hyp']!r}")
print(f"  ref: {literal_worst['ref']!r}")

# A very short gold reference (running header / page number) inflates CER disproportionately --
# one or two wrong characters against a 2-4 character reference is already 50-100% CER, without
# being a meaningful *reading* failure the way a garbled sentence is. Flag this mechanically
# rather than asserting it by eye, so the read below stays honest if the exact worst line changes
# on a re-run.
is_short_reference = len(literal_worst["ref"].split()) <= 3
print(f"\nIs this a short header/page-number-style reference (<=3 words)? {is_short_reference}")

if is_short_reference:
    print(
        "\nRead: the single worst-CER line is a short running-header/page-number fragment, not a "
        "sentence -- a couple of wrong or extra characters against a 2-3 word reference already "
        "produces 50-65% CER, which is a real transcription slip but not the kind of *reading* "
        "failure the CER metric is really meant to flag. It is NOT a conjunct-consonant/matra "
        "segmentation issue -- it looks like a stray page-number/punctuation fragment picked up "
        "alongside the running title, most likely a layout/line-grouping artifact rather than a "
        "character-recognition one."
    )
    substantial = [r for r in worst_lines if len(r["ref"].split()) >= 5]
    if substantial:
        w = substantial[0]
        print(f"\nThe worst line with a *substantial* (>=5 word) reference is more informative "
              f"for OCR reading quality itself:\npage={w['page_id']}  cer={w['cer']:.4f}")
        print(f"  hyp: {w['hyp']!r}")
        print(f"  ref: {w['ref']!r}")
        print(
            "\nRead: this is broad word-level garbling across most of the line, not one isolated "
            "character substitution -- several reference words are missing or replaced outright "
            "rather than merely misspelled. That pattern points more toward local image quality "
            "(blur, ink density, or a damaged/faint region of that specific scan) than a specific "
            "conjunct/matra segmentation rule -- worth visually inspecting that source page region "
            "directly before assuming a linguistic cause, since the text alone doesn't prove one."
        )
else:
    print(
        "\nRead: the worst-CER line already has a substantial reference (>=4 words), so this is "
        "a real reading failure on actual prose, not a short header/page-number artifact -- see "
        "the hyp/ref pair above for what specifically was misread."
    )

Top 5 lines by CER (worst first):

page=tin_sangi_p0217  cer=1.2647
  hyp: যাঁওয়, কেবল আমার পরে, সুন্মির 'পরেও। ওকে স্বাধীন কতৃ-ত্বের সময় দাও
  ref: যাওয়া, কেবল আমার 'পরে নয়, স্থবির

page=tin_sangi_p0219  cer=0.8889
  hyp: তিন সঙ্গী . | ২৩১
  ref: তিন সঙ্গী

page=arogya_p0038  cer=0.6000
  hyp: উদ্দয়ুন
  ref: উদয়ন

page=tin_sangi_p0211  cer=0.4179
  hyp: প্রচলিত নমুলায মানু ও ন। ৯ চর সী ভিড়ের নদে হাটে
  ref: প্রচলিত নমুনার মানুষ ও নয়। ওর নামটা ভিড়ের নামের সঙ্গে হাটে-বাজারে

page=arogya_p0053  cer=0.3333
  hyp: আরোগা
  ref: আরোগ্য

Literal worst line: page=tin_sangi_p0217  cer=1.2647
  hyp: "যাঁওয়, কেবল আমার পরে, সুন্মির 'পরেও। ওকে স্বাধীন কতৃ-ত্বের সময় দাও"
  ref: "যাওয়া, কেবল আমার 'পরে নয়, স্থবির"

Is this a short header/page-number-style reference (<=3 words)? False

Read: the worst-CER line already has a substantial reference (>=4 words), so this is a real reading failure on actual prose, not a short header/page-number artifact -- see the hyp/ref pair above for what spe